# Bias Compare POC

**What this shows:** given 2 Hugging Face toxicity classifiers, we can compare their gender bias *beyond their general sensitivity to wording*. The comparison, the perturbations, and the statistics all live in the `biascompare` package under `src/`; this notebook only calls that package and never reimplements any logic itself.

**Method (one paragraph).** For each post we generate two counterfactuals: a *targeted* gender swap (he/she, brother/sister, ...) and a set of *baseline* synonym swaps that change the same number of words but carry no gender meaning. Following Yang, Levy, Goldberg & Wallace (2026), *"Compared to What? Baselines and Metrics for Counterfactual Prompting"* (arXiv:2605.01048), a demographic swap changing a model's output is **not by itself** evidence of bias, because models react to *any* textual change. We therefore attribute an effect to gender only when the targeted swap moves the model's output distribution (Jensen-Shannon divergence) *more* than the matched-magnitude baseline does — that excess is Δ — and only when Δ is positive more consistently across posts than chance would predict (a one-sided paired t-test), with a companion *direction* statistic (does toxicity rise for the female-referencing version?) and a model-vs-model comparison on top.

**Full design rationale:** every decision made while building this POC, and why, is recorded in [`DESIGN.md`](./DESIGN.md) in this repository — read §6 for the exact statistics and §7 for the exact output schema.


## Setup

Run this cell first. It works unchanged whether you opened this notebook in Google Colab (from GitHub) or are running it locally from a clone of the repository.


In [ ]:
import os, sys

REPO_URL = "https://github.com/shreyasnat2804/bias-compare-poc.git"
REPO_DIR = "/content/bias-compare-poc"

if "google.colab" in sys.modules:
    if os.path.isdir(REPO_DIR):
        !git -C {REPO_DIR} pull -q
    else:
        !git clone -q {REPO_URL} {REPO_DIR}
    %pip install -q -e {REPO_DIR}
    os.chdir(REPO_DIR)

sys.path.insert(0, os.path.abspath("src"))   # import without a runtime restart
# autoreload is optional: some Colab IPython builds still import the removed `imp`
# module on Python 3.12+. Without it, restart the runtime after pulling changes.
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except ModuleNotFoundError:
    print("autoreload unavailable here: restart the runtime after pulling code changes.")


**Notes:**

- Re-run this cell any time to pull the latest code pushed to GitHub.
- If the cell prints that autoreload is unavailable (a known Colab IPython bug on Python 3.12+), restart the runtime after every pull to pick up code changes.
- **Restart the runtime** (Runtime → Restart runtime) after changing a `@dataclass` or a top-level import anywhere in `src/biascompare` — `%autoreload 2` reliably picks up edits to function *bodies*, but not to class shapes or module-level imports.
- An optional Hugging Face access token, set as a Colab Secret named `HF_TOKEN`, is only needed for **gated or private** models. Both POC models are public, so no token is required to run this notebook as-is.


## Inputs

The two models being compared, the dataset, and the run parameters (DESIGN §3).


In [ ]:
from biascompare import ModelSpec

MODELS = [
    ModelSpec("unitary/toxic-bert", toxic_label="toxic"),
    ModelSpec("s-nlp/roberta_toxicity_classifier", toxic_label="toxic"),
]
DATASET = "data/posts.jsonl"
INSTRUCTION = None  # classifiers ignore this; kept for future scorer types (DESIGN §3)
N_CONTROLS = 5
SEED = 0

MODELS, DATASET, INSTRUCTION, N_CONTROLS, SEED


## Preview the perturbations

Before scoring anything, look at what the gender swap and the synonym-swap controls actually did to each post. `preview_perturbations` builds the exact same seeded plan `compare_models` will use, without loading any model.


In [ ]:
import pandas as pd

from biascompare import preview_perturbations

preview = preview_perturbations(DATASET, n_controls=N_CONTROLS, seed=SEED)

preview_df = pd.DataFrame(
    [
        {
            "id": p["id"],
            "gender": p["gender"],
            "original": p["original"],
            "gender_swap": p["gender_swap"]["text"],
            "n_controls": len(p["controls"]),
            "flags": p["flags"],
        }
        for p in preview
    ]
)
preview_df


## Run the comparison

Scores every post's original, gender-swap, and control texts with both models, and computes the per-post metrics, per-model summaries, and model-vs-model comparisons from DESIGN §6. A model that fails to load or score gets an `error` recorded instead of stopping the run.


In [ ]:
from biascompare import compare_models

results = compare_models(
    MODELS,
    DATASET,
    instruction=INSTRUCTION,
    n_controls=N_CONTROLS,
    seed=SEED,
)


## Save results

Writes the full results dict (DESIGN §7) as one JSON file.


In [ ]:
from biascompare import save_results

save_results(results, "results/poc_results.json")


## Verdicts

Plain-English summaries first, then the full numeric summary as a table.


In [ ]:
from biascompare import summary_rows

print(results["summary_verdict"])
print()

for model in results["models"]:
    if model["summary"] is None:
        print(f"{model['name']}: FAILED to score — {model['error']}")
    else:
        print(f"{model['name']}: {model['summary']['verdict']}")

summary_df = pd.DataFrame(summary_rows(results))
summary_df


## Visualisation


In [ ]:
from biascompare import plot_comparison

fig = plot_comparison(results)  # shown inline by the notebook backend


## How to read this

- **Δ (delta)** = the targeted gender swap's effect on the model's output distribution (Jensen-Shannon divergence) *minus* the average effect of the matched-magnitude synonym-swap controls. A Δ near zero means the model reacted to the gender swap the same way it reacts to any comparable wording change — i.e. no gender-specific effect beyond noise. Only a *consistently positive* Δ across posts counts as evidence of gender bias beyond wording sensitivity.
- **Sensitivity** is how much the model's output moves for an irrelevant wording change alone (the mean control JSD) — the baseline every gender effect is measured against.
- **Direction** is signed: positive means the female-referencing version of a post scores *more* toxic than the male-referencing version of the same post, negative means the reverse.
- **n = 10 posts is a small, illustrative sample.** Statistical power is low by design (DESIGN §2, §10) — this POC's job is to prove the *comparison itself* works, not to make a confident claim about either model.
- **"Insufficient evidence" is not "no bias."** With this little data, failing to detect an effect is the expected, honest outcome even if a real effect exists; it should never be read as a finding that a model is unbiased.
